# 09 — Full Evaluation (LOCAL)
## WavSent-MTL · Tasks 4.1–4.16

**Runs on: Local PC**

**What this notebook does:**
- Merge all ablation results (configs A–G) for both datasets
- Aggregate: mean, max, std per config across 30 seeds
- Statistical tests: Wilcoxon (A vs G), Shapiro-Wilk normality check
- Plots: ablation comparison, confusion matrix, AUC-ROC, loss curves
- SHAP analysis on best config
- Granger causality: polarity_mean vs returns (lags 1–5)
- Trading simulation (long-only) + Sharpe ratios
- Baselines: SVM + RF
- Save all figures → results/figures/ and tables → results/tables/

**PREREQUISITE:**
- Download from Kaggle: kotekar_ablation_AG.csv, kaggle_ablation_AG.csv
- Download: pso_weights_*.json, ensemble_results_*.csv
- Place all in ablation/results/kotekar/ and ablation/results/kaggle/
- Download .npy arrays (val_predictions/) from Kaggle outputs

In [ ]:
import sys
import os

project_root = os.path.abspath('..')
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import json
import matplotlib
matplotlib.use('Agg')   # non-interactive backend
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from config.config import CONFIG

# Output directories
for dataset in ['kotekar', 'kaggle']:
    os.makedirs(os.path.join(project_root, CONFIG['figures_dir'], dataset), exist_ok=True)
    os.makedirs(os.path.join(project_root, CONFIG['tables_dir'], dataset), exist_ok=True)

print('CONFIG loaded.')
print('Benchmark: Kotekar et al. accuracy=0.5853, Sharpe=1.5679')

## Task 4.1 — Load and Merge All Ablation Results

In [ ]:
ABLATION_DIR = os.path.join(project_root, CONFIG['ablation_dir'])

results = {}
for dataset in ['kotekar', 'kaggle']:
    # Main ablation CSV (configs A-F)
    main_csv = os.path.join(ABLATION_DIR, dataset, f'{dataset}_ablation_AG.csv')
    df_main = pd.read_csv(main_csv)

    # Config G (ensemble)
    g_csv = os.path.join(ABLATION_DIR, dataset, f'ensemble_results_{dataset}.csv')
    # Also check working directory from Kaggle download
    if not os.path.exists(g_csv):
        g_csv = os.path.join(project_root, f'ensemble_results_{dataset}.csv')
    df_g = pd.read_csv(g_csv) if os.path.exists(g_csv) else pd.DataFrame()

    df_all = pd.concat([df_main, df_g], ignore_index=True)
    results[dataset] = df_all

    print(f'{dataset}: {len(df_all)} rows | configs: {sorted(df_all["config"].unique())}')

results['kotekar'].head(3)

## Task 4.2 — Aggregate: Mean, Max, Std per Config

In [ ]:
from src.evaluation.metrics import aggregate_run_metrics

METRIC_COLS = ['accuracy', 'balanced_accuracy', 'auc', 'precision', 'recall', 'f1']

for dataset in ['kotekar', 'kaggle']:
    df = results[dataset]
    rows = []
    for cfg in sorted(df['config'].unique()):
        cfg_df = df[df['config'] == cfg]
        for metric in METRIC_COLS:
            if metric in cfg_df.columns:
                rows.append({
                    'config': cfg,
                    'metric': metric,
                    'mean':   cfg_df[metric].mean(),
                    'max':    cfg_df[metric].max(),
                    'std':    cfg_df[metric].std(),
                })

    summary_df = pd.DataFrame(rows)
    out_path = os.path.join(project_root, CONFIG['tables_dir'], dataset, 'ablation_summary.csv')
    summary_df.to_csv(out_path, index=False)
    print(f'{dataset} summary saved: {out_path}')

    # Print accuracy summary
    acc_summary = summary_df[summary_df['metric'] == 'accuracy'][['config','mean','max','std']]
    print(f'\n{dataset.upper()} — Accuracy across configs:')
    print(acc_summary.to_string(index=False))
    print()

## Task 4.3 — Wilcoxon Signed-Rank Test (Config A vs Config G)

In [ ]:
from scipy.stats import wilcoxon, shapiro

wilcoxon_results = {}

for dataset in ['kotekar', 'kaggle']:
    df = results[dataset]
    acc_A = df[df['config'] == CONFIG['wilcoxon_config_a']]['accuracy'].values
    acc_G = df[df['config'] == CONFIG['wilcoxon_config_b']]['accuracy'].values

    print(f'\n--- {dataset.upper()} ---')

    # Task 4.4: Shapiro-Wilk normality check
    if len(acc_A) >= 3:
        stat_a, p_a = shapiro(acc_A)
        stat_g, p_g = shapiro(acc_G) if len(acc_G) >= 3 else (None, None)
        print(f'Shapiro-Wilk Config A: stat={stat_a:.4f}, p={p_a:.4f} -> {"normal" if p_a > 0.05 else "non-normal"}')
        if p_g is not None:
            print(f'Shapiro-Wilk Config G: stat={stat_g:.4f}, p={p_g:.4f} -> {"normal" if p_g > 0.05 else "non-normal"}')

    if len(acc_A) >= 2 and len(acc_G) >= 1:
        if len(acc_A) == len(acc_G):
            stat, p = wilcoxon(acc_A, acc_G)
        else:
            # Config G has only 1 run — compare A distribution vs G single value
            from scipy.stats import ttest_1samp
            stat, p = ttest_1samp(acc_A, acc_G[0])
            print('(Note: Using 1-sample t-test since Config G has 1 run)')

        significance = 'SIGNIFICANT' if p < 0.05 else 'not significant'
        print(f'Wilcoxon A vs G: stat={stat:.4f}, p={p:.4f} -> {significance}')
        wilcoxon_results[dataset] = {'stat': float(stat), 'p_value': float(p), 'significant': p < 0.05}

# Save
for dataset in wilcoxon_results:
    out = os.path.join(project_root, CONFIG['tables_dir'], dataset, 'granger_results.csv')
    pd.DataFrame([wilcoxon_results[dataset]]).to_csv(out, index=False)

## Task 4.5 — Ablation Comparison Plot

In [ ]:
for dataset in ['kotekar', 'kaggle']:
    df = results[dataset]
    cfg_order = [c for c in ['A','B','C','D','E','F','G'] if c in df['config'].unique()]
    means = [df[df['config']==c]['accuracy'].mean() for c in cfg_order]
    stds  = [df[df['config']==c]['accuracy'].std() for c in cfg_order]

    fig, ax = plt.subplots(figsize=(10, 5))
    bars = ax.bar(cfg_order, means, yerr=stds, capsize=4,
                  color=['#90CAF9']*3 + ['#4CAF50']*3 + ['#E53935'],
                  edgecolor='black', linewidth=0.5)
    ax.axhline(0.5853, color='red', linestyle='--', linewidth=1.5,
               label='Kotekar benchmark (0.5853)')
    ax.set_xlabel('Configuration')
    ax.set_ylabel('Test Accuracy')
    ax.set_title(f'Ablation Study — {dataset.capitalize()} Dataset')
    ax.legend()
    ax.set_ylim(0.45, max(means) + 0.05)

    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f'{m:.3f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    out_path = os.path.join(project_root, CONFIG['figures_dir'], dataset, 'ablation_comparison.png')
    fig.savefig(out_path, dpi=150)
    plt.close()
    print(f'Saved: {out_path}')

## Task 4.6 — Confusion Matrix (Config G, Kotekar)

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

for dataset in ['kotekar', 'kaggle']:
    # Load ensemble test predictions
    pred_dir = os.path.join(ABLATION_DIR, dataset, 'val_predictions')
    weights_path = os.path.join(ABLATION_DIR, dataset, f'pso_weights_{dataset}.json')
    test_npy = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'], 'y_clf_test.npy')

    if not os.path.exists(weights_path) or not os.path.exists(test_npy):
        print(f'{dataset}: skipping confusion matrix (files not found)')
        continue

    with open(weights_path) as f:
        weights = json.load(f)

    y_test = np.load(test_npy)

    # Reconstruct ensemble test predictions
    ensemble_probs = np.zeros(len(y_test), dtype=np.float32)
    for model_name in CONFIG['pso_models']:
        test_pred_path = os.path.join(pred_dir, f'{model_name}_test_preds.npy')
        if os.path.exists(test_pred_path):
            ensemble_probs += weights.get(model_name, 0) * np.load(test_pred_path)

    y_pred = (ensemble_probs >= 0.5).astype(int)
    cm = confusion_matrix(y_test, y_pred)

    fig, ax = plt.subplots(figsize=(5, 4))
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Down', 'Up'])
    disp.plot(ax=ax, colorbar=False)
    ax.set_title(f'Confusion Matrix — Config G — {dataset.capitalize()}')
    plt.tight_layout()
    out_path = os.path.join(project_root, CONFIG['figures_dir'], dataset, 'confusion_matrix.png')
    fig.savefig(out_path, dpi=150)
    plt.close()
    print(f'Saved: {out_path}')

## Task 4.7 — AUC-ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, auc

for dataset in ['kotekar', 'kaggle']:
    pred_dir = os.path.join(ABLATION_DIR, dataset, 'val_predictions')
    weights_path = os.path.join(ABLATION_DIR, dataset, f'pso_weights_{dataset}.json')
    test_npy = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'], 'y_clf_test.npy')

    if not os.path.exists(weights_path) or not os.path.exists(test_npy):
        print(f'{dataset}: skipping AUC-ROC (files not found)')
        continue

    with open(weights_path) as f:
        weights = json.load(f)
    y_test = np.load(test_npy)

    fig, ax = plt.subplots(figsize=(7, 6))

    # Individual model curves
    colors = {'tkan': '#1976D2', 'lstm': '#388E3C', 'gru': '#F57C00', 'tcn': '#7B1FA2'}
    for model_name in CONFIG['pso_models']:
        pred_path = os.path.join(pred_dir, f'{model_name}_test_preds.npy')
        if os.path.exists(pred_path):
            probs = np.load(pred_path)
            fpr, tpr, _ = roc_curve(y_test, probs)
            roc_auc = auc(fpr, tpr)
            ax.plot(fpr, tpr, color=colors.get(model_name, 'gray'),
                    label=f'{model_name.upper()} (AUC={roc_auc:.3f})', linewidth=1.2)

    # Ensemble curve
    ensemble_probs = np.zeros(len(y_test), dtype=np.float32)
    for model_name in CONFIG['pso_models']:
        pred_path = os.path.join(pred_dir, f'{model_name}_test_preds.npy')
        if os.path.exists(pred_path):
            ensemble_probs += weights.get(model_name, 0) * np.load(pred_path)

    fpr_g, tpr_g, _ = roc_curve(y_test, ensemble_probs)
    auc_g = auc(fpr_g, tpr_g)
    ax.plot(fpr_g, tpr_g, color='red', linewidth=2.5,
            label=f'Config G / Ensemble (AUC={auc_g:.3f})')
    ax.plot([0,1], [0,1], 'k--', alpha=0.4)
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'AUC-ROC Curves — {dataset.capitalize()}')
    ax.legend(loc='lower right', fontsize=9)
    plt.tight_layout()
    out_path = os.path.join(project_root, CONFIG['figures_dir'], dataset, 'auc_roc_curve.png')
    fig.savefig(out_path, dpi=150)
    plt.close()
    print(f'Saved: {out_path}')

## Task 4.9 — SHAP Analysis on Best Config

In [ ]:
import torch
from src.models.mtl_model import build_model
from src.evaluation.shap_analysis import run_shap_analysis

device = 'cpu'

for dataset in ['kotekar', 'kaggle']:
    proc_dir = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'])
    feat_path = os.path.join(proc_dir, 'selected_features.json')

    if not os.path.exists(feat_path):
        print(f'{dataset}: selected_features.json not found, skipping SHAP')
        continue

    with open(feat_path) as f:
        selected_features = json.load(f)

    X_train = np.load(os.path.join(proc_dir, 'X_train.npy'))
    X_val   = np.load(os.path.join(proc_dir, 'X_val.npy'))
    n_features = X_train.shape[2]

    # Build best model (BEST_REPR config = C or B, TKAN)
    # Use TKAN for SHAP (best config model)
    BEST_REPR = CONFIG['BEST_REPR']
    model_name = 'tkan'  # best individual model from ablation
    model = build_model(model_name, CONFIG, n_features)

    # Try to load saved model weights
    model_path = os.path.join(project_root, CONFIG['models_dir'], dataset, f'{model_name}_best.pt')
    if os.path.exists(model_path):
        model.load_state_dict(torch.load(model_path, map_location='cpu'))
        print(f'Loaded saved model: {model_path}')
    else:
        print(f'No saved model at {model_path} — using random weights for SHAP demo')

    model.eval()

    print(f'Running SHAP analysis: {dataset} | n_features={n_features}')
    shap_vals = run_shap_analysis(
        model=model,
        X_train=X_train,
        X_explain=X_val,
        feature_names=selected_features,
        dataset=dataset,
        n_background=min(100, len(X_train)),
        n_explain=min(200, len(X_val)),
        save_fig=True,
    )
    print(f'SHAP values shape: {shap_vals.shape}')

## Task 4.10 — Granger Causality Test

In [ ]:
from statsmodels.tsa.stattools import grangercausalitytests

MAX_LAG = CONFIG['granger_max_lag']

for dataset in ['kotekar', 'kaggle']:
    proc_dir = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'])
    feat_csv = os.path.join(proc_dir, 'featured_data.csv')

    if not os.path.exists(feat_csv):
        print(f'{dataset}: featured_data.csv not found, skipping Granger')
        continue

    df_feat = pd.read_csv(feat_csv)
    df_feat['return'] = df_feat['Close'].pct_change()
    df_granger = df_feat[['return', 'polarity_mean']].dropna()

    print(f'\n--- {dataset.upper()} — Granger Causality: polarity_mean -> returns ---')
    gc_results = grangercausalitytests(
        df_granger[['return', 'polarity_mean']].values,
        maxlag=MAX_LAG, verbose=False
    )

    gc_rows = []
    for lag in range(1, MAX_LAG + 1):
        f_stat = gc_results[lag][0]['ssr_ftest'][0]
        p_val  = gc_results[lag][0]['ssr_ftest'][1]
        sig = 'Yes' if p_val < 0.05 else 'No'
        print(f'  Lag {lag}: F={f_stat:.4f}  p={p_val:.4f}  significant={sig}')
        gc_rows.append({'lag': lag, 'f_stat': f_stat, 'p_value': p_val, 'significant': sig})

    gc_df = pd.DataFrame(gc_rows)
    out_path = os.path.join(project_root, CONFIG['tables_dir'], dataset, 'granger_results.csv')
    gc_df.to_csv(out_path, index=False)
    print(f'Saved: {out_path}')

## Tasks 4.11–4.12 — Trading Simulation + Sharpe

In [ ]:
from src.evaluation.trading_sim import run_trading_simulation

trading_results = {}

for dataset in ['kotekar', 'kaggle']:
    pred_dir = os.path.join(ABLATION_DIR, dataset, 'val_predictions')
    weights_path = os.path.join(ABLATION_DIR, dataset, f'pso_weights_{dataset}.json')
    proc_dir = os.path.join(project_root, CONFIG[f'{dataset}_processed_dir'])
    feat_csv = os.path.join(proc_dir, 'featured_data.csv')

    if not os.path.exists(weights_path) or not os.path.exists(feat_csv):
        print(f'{dataset}: skipping trading sim (files not found)')
        continue

    with open(weights_path) as f:
        weights = json.load(f)

    df_feat = pd.read_csv(feat_csv)
    y_test  = np.load(os.path.join(proc_dir, 'y_clf_test.npy'))

    # Get test period close prices from temporal split
    from src.data.windows import temporal_split
    df_feat['Date'] = pd.to_datetime(df_feat['Date'])
    _, _, test_df = temporal_split(df_feat)
    close_test = test_df['Close'].values

    # Reconstruct ensemble test predictions
    ensemble_probs = np.zeros(len(y_test), dtype=np.float32)
    for model_name in CONFIG['pso_models']:
        pred_path = os.path.join(pred_dir, f'{model_name}_test_preds.npy')
        if os.path.exists(pred_path):
            ensemble_probs += weights.get(model_name, 0) * np.load(pred_path)

    print(f'\n--- {dataset.upper()} Trading Simulation (Config G) ---')
    sim_result = run_trading_simulation(
        y_prob=ensemble_probs,
        close_prices=close_test,
        dataset=dataset,
        config_name='G',
        save_fig=True,
    )
    trading_results[dataset] = sim_result

    # Save
    out_path = os.path.join(project_root, CONFIG['tables_dir'], dataset, 'trading_results.csv')
    pd.DataFrame([sim_result]).to_csv(out_path, index=False)
    print(f'Saved: {out_path}')

## Task 4.15 — Baselines (SVM + RF)

In [ ]:
import subprocess
print('Running baselines...')
result = subprocess.run(
    ['python', os.path.join(project_root, 'baselines', 'run_baselines.py')],
    capture_output=True, text=True, cwd=project_root
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)

## Final Summary vs Kotekar Benchmark

In [ ]:
KOTEKAR_BENCHMARK_ACC   = 0.5853
KOTEKAR_BENCHMARK_SHARPE = 1.5679

print('=' * 65)
print('FINAL RESULTS vs BENCHMARK')
print('=' * 65)

for dataset in ['kotekar', 'kaggle']:
    df = results[dataset]
    cfg_G = df[df['config'] == 'G']
    if len(cfg_G) > 0:
        acc_G = cfg_G['accuracy'].values[0]
        auc_G = cfg_G['auc'].values[0]
    else:
        acc_G = auc_G = None

    sharpe_G = trading_results.get(dataset, {}).get('sharpe', None)

    print(f'\n{dataset.upper()}:')
    print(f'  Config G accuracy:  {acc_G:.4f}' if acc_G else '  Config G: not available')
    print(f'  Benchmark accuracy: {KOTEKAR_BENCHMARK_ACC}')
    if acc_G:
        delta = acc_G - KOTEKAR_BENCHMARK_ACC
        print(f'  Delta:              {delta:+.4f}  {"BEAT" if delta > 0 else "below"}')
    if sharpe_G:
        print(f'  Config G Sharpe:    {sharpe_G:.4f}')
        print(f'  Benchmark Sharpe:   {KOTEKAR_BENCHMARK_SHARPE}')

print()
print('All figures saved to:   results/figures/')
print('All tables saved to:    results/tables/')
print()
print('Phase 4 COMPLETE — ready for paper writing (Phase 5)')